In [0]:
%sql
-- VISTA PARA DISEÑAR LA TABLA SILVER 
DROP VIEW IF EXISTS ventas_bronze_EDA;

CREATE OR REPLACE VIEW ventas_bronze_EDA AS

WITH limpieza AS (
    SELECT
        venta,
        articulo,

        -- Limpieza basica
        TRIM(
            regexp_replace(
                regexp_replace(
                    regexp_replace(
                        TRANSLATE(lower(descrip),
                                  'áéíóúüñ',
                                  'aeiouun'),
                        '[\".]', ' '
                    ),
                    '[()]', ''
                ),
                '\\s+', ' '
            )
        ) AS clean_descrip,

        TRY_CAST(REGEXP_REPLACE(precio, ',', '.') AS DOUBLE) AS precio,
        TRY_CAST(cant AS INT) AS cant,
        TRY_CAST(REGEXP_REPLACE(total, ',', '.') AS DOUBLE) AS total,

        LOWER(TRIM(usulogin)) AS usulogin,
        LOWER(TRIM(clinombre)) AS clinombre,
        --cliente,
        vtaoperacion,
        LOWER(TRIM(vtaestado)) AS vtaestado,
        TO_TIMESTAMP(vtafecha, 'd/M/yyyy HH:mm') AS vtafecha,
        condvtapos,
        LOWER(TRIM(delivery)) AS delivery,
        --turno,
        --caja,
        comprobante
        --LOWER(TRIM(succodigo)) AS succodigo
        --sucursal 


    FROM catalog_ventas.raw.ventas_heladeria_bronze
   
)

SELECT
    venta,
    articulo,

    -- Estandarización + eliminación de "envios a domicilio"
    CASE 
        WHEN clean_descrip LIKE '%1/2 kilo%' THEN '1/2kilo'
        WHEN clean_descrip LIKE '%1 kilo%' THEN '1kilo'
        WHEN clean_descrip LIKE '%1/4 kilo%' THEN '1/4kilo'
        WHEN clean_descrip LIKE '%sabores%' THEN '1/4kilo'
        ELSE 
        NULLIF(
            TRIM(
                regexp_replace(
                    regexp_replace(
                        clean_descrip,
                        'envios? a domicilio|envio|domicilio|delivery',
                        ''
                    ),
                    '\\s+',
                    ' '
                )
            ),
            ''
        )
    END AS descrip,

    precio,
    cant,
    total,
    usulogin,
    
    -- Estandarizacion de clientes
    CASE 
        WHEN vtaoperacion = 'VL' OR vtaoperacion='CL' THEN 'socio'
        WHEN vtaoperacion = 'VF' THEN 'no_socio'
        WHEN vtaoperacion = 'VG' THEN 'gastronomico'
    END AS clientes,
    
    vtaestado,
    vtafecha,

    --Normalizar turno (mañana/noche)
    CASE    
        WHEN HOUR(vtafecha) BETWEEN 10 AND 18 THEN 'mañana'
        ELSE 'noche' 
    END AS  turno,

    --Estandarizacion de medios de pago
    CASE 
        WHEN condvtapos='TD' THEN 'debito'
        WHEN condvtapos='TC' THEN 'credito'
        WHEN condvtapos='EF' THEN 'efectivo'
        WHEN condvtapos='MU' THEN 'multiple_opciones' 
        WHEN condvtapos='XX' THEN 'cancelado'
        ELSE '0'
    END AS condvtapos,

   --Normalizar valores (si/no) 
   CASE 
        WHEN delivery IN ('si','1','true') THEN 1
        ELSE 0
    END AS delivery,

    --Normalizar valores nulls de comprobante
    CASE WHEN comprobante IS NULL THEN 'sin_comprobante' ELSE comprobante END AS comprobante
    
    --succodigo
    --turno,
    --clinombre,
    --cliente,
    --caja,

FROM limpieza;

In [0]:
%sql
-- VERIFICAR DATOS DISPONIBLES
SELECT *
FROM ventas_bronze_EDA LIMIT 10


In [0]:
%sql
-- VERIFICAR ESTRUCTURA
DESCRIBE ventas_bronze_EDA